# 11 - RAPIDS cuDF: alternativa GPU a pandas (con comparativa)

En este notebook:

- Introducción a **cuDF** (RAPIDS), una alternativa compatible con gran parte de la API de pandas, pero acelerada en GPU.
- Crear DataFrames, seleccionar, filtrar y agrupar con cuDF.
- Comparar tiempos sencillos con pandas (si hay GPU).

> **Nota:** Este cuaderno intenta importar `cudf`. Si no está disponible, las celdas mostrarán un mensaje y puedes ejecutarlas en un entorno con GPU (por ejemplo, Colab con RAPIDS).


## 1. Importación de cuDF y dataset sintético

In [ ]:
import numpy as np
import pandas as pd

try:
    import cudf
    has_cudf = True
    print("cuDF disponible")
except Exception as e:
    has_cudf = False
    print("cuDF NO disponible en este entorno:", e)

rng = np.random.default_rng(0)
n = 1_00000  # 100k filas
pdf = pd.DataFrame({
    "id": np.arange(n),
    "x": rng.normal(0,1,size=n),
    "y": rng.integers(0, 10, size=n),
    "city": rng.choice(["Cádiz","Sevilla","Málaga","Córdoba"], size=n)
})
pdf.head()

## 2. Operaciones básicas en cuDF

In [ ]:
if has_cudf:
    gdf = cudf.from_pandas(pdf)
    # Selección, filtrado, columna nueva
    gdf2 = gdf[gdf["y"] > 5].assign(z = gdf["x"]**2)
    gdf2.head()
else:
    print("Salta esta celda si no tienes cuDF.")

## 3. GroupBy y agregaciones

In [ ]:
if has_cudf:
    agg = gdf.groupby("city").agg({"x":"mean","y":"mean"}).reset_index()
    agg
else:
    print("Salta esta celda si no tienes cuDF.")

## 4. Comparativa de tiempos (pandas vs cuDF)

In [ ]:
import time

def pandas_pipeline(df):
    return (df[df["y"] > 3]
            .assign(z = df["x"]**2)
            .groupby("city")["z"]
            .mean()
            .sort_values(ascending=False))

t0 = time.time()
out_pd = pandas_pipeline(pdf.copy())
t1 = time.time()

print(f"Tiempo pandas: {t1-t0:.4f}s")
print(out_pd.head())

if has_cudf:
    def cudf_pipeline(df):
        return (df[df["y"] > 3]
                .assign(z = df["x"]**2)
                .groupby("city")["z"]
                .mean()
                .sort_values(ascending=False))

    gdf = cudf.from_pandas(pdf)
    t2 = time.time()
    out_gd = cudf_pipeline(gdf)
    t3 = time.time()
    print(f"Tiempo cuDF:  {t3-t2:.4f}s")
    print(out_gd.head())
else:
    print("cuDF no disponible: comparativa parcial con pandas.")

## 5. Interoperabilidad pandas ↔ cuDF

In [ ]:
if has_cudf:
    # pandas -> cuDF
    gdf = cudf.from_pandas(pdf)
    # cuDF -> pandas
    back_to_pandas = gdf.to_pandas()
    type(gdf), type(back_to_pandas)
else:
    print("Sin cuDF instalado, omite esta celda.")

## 📝 Ejercicios

1) Crea una columna `w = x*y` en cuDF y calcula la media de `w` por `city`.  
2) Replica la misma transformación en pandas y compara resultados.  
3) Aumenta `n` a 1 millón si tu GPU lo permite y repite la comparativa.
